# 04 学习数据预处理

本节目标：用 pandas 读取数据，处理缺失值和类别变量，做数值归一化，并把 DataFrame 转成 PyTorch 张量。

## 1. 最小概念

- 读取数据：用 `pandas.read_csv()` 读取 CSV 表格。
- 缺失值：真实数据里常有空值，需要填充或删除。
- 类别变量：文本类别不能直接喂给模型，通常要转成数字。
- 数值归一化：把不同范围的数值特征缩放到更接近的范围。
- 转张量：模型训练最终通常使用 PyTorch 张量作为输入。

In [ ]:
from pathlib import Path

import pandas as pd
import torch


project_root = Path.cwd()
if project_root.name == "playground":
    project_root = project_root.parent

data_dir = project_root / "data"
data_dir.mkdir(exist_ok=True)
data_file = data_dir / "house_tiny.csv"

csv_text = """NumRooms,Age,Neighborhood,Alley,Price
NA,12,North,Pave,127500
2,NA,South,NA,106000
4,8,North,NA,178100
NA,20,West,Gravel,140000
3,15,South,Pave,156000
"""

data_file.write_text(csv_text, encoding="utf-8")
print("CSV 文件位置:", data_file)

## 2. 读取 CSV

`NA` 表示缺失值。读取时用 `na_values="NA"` 把它识别成 pandas 的缺失值。

In [ ]:
data = pd.read_csv(data_file, na_values="NA")

print("原始数据:")
print(data)
print("\n每列缺失值数量:")
print(data.isna().sum())

## 3. 拆分输入特征和标签

通常把模型要看的数据叫输入特征 `X`，把模型要预测的结果叫标签 `y`。这里我们用房屋信息预测价格。

In [ ]:
inputs = data.drop(columns="Price")
labels = data["Price"]

print("输入特征:")
print(inputs)
print("\n标签:")
print(labels)

## 4. 处理缺失值

- 数值列：用该列平均值填充。
- 类别列：用 `Unknown` 填充，表示未知类别。

In [ ]:
numeric_cols = inputs.select_dtypes(include="number").columns
category_cols = inputs.select_dtypes(exclude="number").columns

clean_inputs = inputs.copy()
clean_inputs[numeric_cols] = clean_inputs[numeric_cols].fillna(clean_inputs[numeric_cols].mean())
clean_inputs[category_cols] = clean_inputs[category_cols].fillna("Unknown")

print("清洗后的输入:")
print(clean_inputs)
print("\n缺失值数量:")
print(clean_inputs.isna().sum())

## 5. 数值归一化

不同数值列的范围可能差很多。这里使用标准化：

`新值 = (原值 - 平均值) / 标准差`

标准化后，数值列会更容易被模型学习。

In [ ]:
normalized_inputs = clean_inputs.copy()

means = normalized_inputs[numeric_cols].mean()
stds = normalized_inputs[numeric_cols].std(ddof=0).replace(0, 1)
normalized_inputs[numeric_cols] = (normalized_inputs[numeric_cols] - means) / stds

print("数值归一化后的输入:")
print(normalized_inputs)
print("\n数值列均值:")
print(normalized_inputs[numeric_cols].mean().round(6))
print("\n数值列标准差:")
print(normalized_inputs[numeric_cols].std(ddof=0).round(6))

## 6. 类别变量编码

模型不能直接理解 `North`、`South`、`Pave` 这样的字符串。这里用独热编码，把每个类别变成 0/1 数字列。

In [ ]:
encoded_inputs = pd.get_dummies(normalized_inputs, columns=category_cols, dtype="float32")

print("编码后的输入:")
print(encoded_inputs)
print("\n编码后 shape:", encoded_inputs.shape)

## 7. 从 DataFrame 转 PyTorch 张量

清洗、归一化和编码完成后，就可以把表格转成张量，作为模型输入。

In [ ]:
X = torch.tensor(encoded_inputs.to_numpy(dtype="float32"))
y = torch.tensor(labels.to_numpy(dtype="float32")).reshape(-1, 1)

print("X:")
print(X)
print("X shape:", tuple(X.shape))
print("X dtype:", X.dtype)

print("\ny:")
print(y)
print("y shape:", tuple(y.shape))
print("y dtype:", y.dtype)

## 小结

数据预处理的一条基础流程：读取 CSV -> 识别缺失值 -> 拆分特征和标签 -> 填充缺失值 -> 数值归一化 -> 类别编码 -> 转成张量。